In [7]:
trade={
    "symbol":"ETH/USDT",
    "direction":"long",
    "entry_price":2100.00,
    "exit_price":2200.00,
    "quantity":0.5,
    "stop_loss":2050.00,
    "take_profit":2300.00,
    "entry_time":"2026-05-20 09:00",
    "exit_time":"2026-05-21 15:30"
}

pnl=(trade['exit_price']-trade['entry_price'])*trade['quantity']
trade['pnl']=pnl
trade['pnl_pct']=pnl/(trade['entry_price']*trade['quantity'])*100

strategy_params={
    'name':"EMA_crossover",
    'fast_period':20,
    'slow_period':50,
    'rsi_period':14,
    'risk_per_trade':0.02,
    'max_leverage':3
}

trade_log= [
    {"date":"2026-05-20","direction":"long","pnl":50.0},
    {"date":"2026-05-21","direction":"short","pnl":-20.0},
    {"date":"2026-05-22","direction":"long","pnl":80.0},
    {"date":"2026-05-23","direction":"long", "pnl":-30.0},
    {"date":"2026-05-24","direction":"short","pnl":60.0},
]

total_pnl=sum(t['pnl'] for t in trade_log)
wins=[t for t in trade_log if t['pnl']>0]
losses=[t for t in trade_log if t['pnl'] <= 0]
win_rate=len(wins)/len(trade_log)*100

print(f"交易统计:")
print(f"总笔数:{len(trade_log)}")
print(f"盈利:{len(wins)}笔/亏损:{len(losses)}笔")
print(f"胜率:{win_rate:.1f}%")
print(f"总盈亏:{total_pnl:.2f}USD")

交易统计:
总笔数:5
盈利:3笔/亏损:2笔
胜率:60.0%
总盈亏:140.00USD


In [ ]:
def calculate_position_size(
    account_balance:float,
    risk_pct:float,
    entry_price:float,
    stop_loss:float
)->float:
    max_loss=account_balance*risk_pct
    price_risk=abs(entry_price-stop_loss)

    if price_risk==0:
        return 0
    position_size=max_loss/price_risk
    return position_size

def risk_reward_ratio(
    entry_price:float,
    stop_loss:float,
    take_profit:float
)->float:
    risk=abs(entry_price-stop_loss)
    reward=abs(take_profit-entry_price)
    
    if risk==0:
        return 0.0
    return reward/risk

def calculate_expectancy(trades:list[dict])->dict:
    
    if not trades:
        return {"error":"无交易记录"}
    
    wins=[t['pnl'] for t in trades if t['pnl']>0]
    losses=[t['pnl'] for t in trades if t['pnl']<=0]
    win_rate=len(wins)/len(trades)
    avg_win=sum(wins)/len(wins) if wins else 0
    avg_loss=abs(sum(losses)/len(losses)) if losses else 0
    
    expectancy=(win_rate*avg_win)-((1-win_rate)*avg_loss)
    
    return {
        "total_trades":len(trades),
        "win_rate":win_rate,
        "avg_win":avg_win,
        "avg_loss":avg_loss,
        "profit_factor":avg_win/avg_loss if avg_loss>0 else float('inf'),
        "expectancy":expectancy,
        "expectancy_per_dollar":expectancy/avg_loss if avg_loss>0 else 0
    }

    
    